In [1]:
# ================================
# 1. IMPORTS & SEED
# ================================

import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

from skimage.feature import hog
import lightgbm as lgb
import joblib

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

In [2]:
import shutil

shutil.make_archive('/kaggle/input/datasets/thilak02/breast-cancer-detection-using-thermography/BCD_Dataset', 'zip', '/kaggle/input')

OSError: [Errno 30] Read-only file system: '/kaggle/input/datasets/thilak02/breast-cancer-detection-using-thermography/BCD_Dataset.zip'

In [ ]:
# ================================
# 2. LOAD DATA
# ================================

IMG_SIZE = 224
dataset_path = "/kaggle/input/datasets/thilak02/breast-cancer-detection-using-thermography/BCD_Dataset"

images, labels = [], []

for cls, label in [("normal", 0), ("Sick", 1)]:
    path = os.path.join(dataset_path, cls)

    for img in os.listdir(path):
        image = cv2.imread(os.path.join(path, img))
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        images.append(image)
        labels.append(label)

images = np.array(images) / 255.0
labels = np.array(labels)

print("Dataset:", images.shape)

In [ ]:
# ================================
# 3. SPLIT DATA
# ================================

X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, stratify=labels, random_state=42
)

In [ ]:
# ================================
# 4. HOG FEATURES
# ================================

def extract_features(images):
    feats = []

    for img in images:
        gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_BGR2GRAY)

        hog_feat = hog(gray,
                       orientations=9,
                       pixels_per_cell=(16,16),
                       cells_per_block=(2,2),
                       feature_vector=True)

        mean = np.mean(img, axis=(0,1))
        std  = np.std(img, axis=(0,1))

        feats.append(np.hstack([hog_feat, mean, std]))

    return np.array(feats)

X_train_feat = extract_features(X_train)
X_test_feat  = extract_features(X_test)

scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat  = scaler.transform(X_test_feat)

In [ ]:
# ================================
# 5. CNN + ATTENTION
# ================================

class CNN_Attention(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )

        self.attn = nn.Sequential(
            nn.Conv2d(128,128,1),
            nn.Sigmoid()
        )

        self.fc = nn.Sequential(
            nn.Linear(128*28*28,128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128,2)
        )

    def forward(self,x):
        x = self.features(x)
        x = x * self.attn(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [ ]:
def to_tensor(X,y):
    X = torch.tensor(X.transpose(0,3,1,2), dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.long)
    return TensorDataset(X,y)

train_loader = DataLoader(to_tensor(X_train,y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(to_tensor(X_test,y_test), batch_size=32)

In [ ]:
model = CNN_Attention().to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(15):
    model.train()
    total_loss = 0

    for xb,yb in train_loader:
        xb,yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: {total_loss:.4f}")

In [ ]:
def extract_embeddings(model, loader):
    model.eval()
    embs = []

    with torch.no_grad():
        for xb,_ in loader:
            xb = xb.to(DEVICE)
            x = model.features(xb)
            x = x.mean([2,3])
            embs.append(x.cpu().numpy())

    return np.vstack(embs)

emb_train = extract_embeddings(model, train_loader)
emb_test  = extract_embeddings(model, test_loader)

In [ ]:
# ================================
# FEATURE FUSION + CLEANING
# ================================

Xh_train = np.hstack([X_train_feat, emb_train * 2])
Xh_test  = np.hstack([X_test_feat,  emb_test * 2])

# Remove constant features
from sklearn.feature_selection import VarianceThreshold

var_selector = VarianceThreshold(threshold=0.0)
Xh_train = var_selector.fit_transform(Xh_train)
Xh_test  = var_selector.transform(Xh_test)

# Select best features
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(f_classif, k=200)
Xh_train = selector.fit_transform(Xh_train, y_train)
Xh_test  = selector.transform(Xh_test)

In [ ]:
clf = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=20,
    min_data_in_leaf=3,
    max_depth=5,
    verbosity=-1
)

# ================================
# CONVERT TO DATAFRAME (ADD HERE)
# ================================

import pandas as pd


Xh_train = pd.DataFrame(Xh_train)
Xh_test  = pd.DataFrame(Xh_test)

clf.fit(Xh_train, y_train)
pred = clf.predict(Xh_test)

In [ ]:
print("\nClassification Report:")
print(classification_report(y_test, pred))

print("Accuracy:", accuracy_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

probs = clf.predict_proba(Xh_test)[:,1]
print("ROC-AUC:", roc_auc_score(y_test, probs))

In [ ]:
print("\n=== Ablation Study ===")

# HOG
clf_hog = lgb.LGBMClassifier(verbosity=-1)
clf_hog.fit(X_train_feat, y_train)
pred_hog = clf_hog.predict(X_test_feat)

# CNN
clf_cnn = lgb.LGBMClassifier(verbosity=-1)
clf_cnn.fit(emb_train, y_train)
pred_cnn = clf_cnn.predict(emb_test)

print("HOG:", accuracy_score(y_test, pred_hog))
print("CNN:", accuracy_score(y_test, pred_cnn))
print("Proposed:", accuracy_score(y_test, pred))

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, Xh_train, y_train, cv=cv)

print("CV Accuracy:", scores.mean())

In [ ]:
joblib.dump(clf, "model.pkl")

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

print("\n=== ROC-AUC ===")

probs = clf.predict_proba(Xh_test)[:, 1]

auc = roc_auc_score(y_test, probs)
print("ROC-AUC:", auc)

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_test, probs)

plt.figure()
plt.plot(fpr, tpr)
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("\n=== CONFUSION MATRIX ===")

cm = confusion_matrix(y_test, pred)
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()

In [ ]:
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        # register hooks
        self.target_layer.register_forward_hook(self.forward_hook)
        self.target_layer.register_full_backward_hook(self.backward_hook)

    def forward_hook(self, module, input, output):
        self.activations = output

    def backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()

        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = torch.argmax(output)

        self.model.zero_grad()
        output[0, class_idx].backward()

        gradients = self.gradients
        activations = self.activations

        # Global Average Pooling (weights)
        weights = torch.mean(gradients, dim=(2, 3), keepdim=True)

        # Weighted combination
        cam = torch.sum(weights * activations, dim=1)

        cam = cam.squeeze().detach().cpu().numpy()

        # ReLU
        cam = np.maximum(cam, 0)

        # Normalize
        cam = cam / (cam.max() + 1e-8)

        return cam

In [ ]:
target_layer = model.features[-3]  # last conv layer
gradcam = GradCAM(model, target_layer)

In [ ]:
def visualize_gradcam(original_img, cam):

    cam = cv2.resize(cam, (224, 224))

    heatmap = cv2.applyColorMap(
        np.uint8(255 * cam),
        cv2.COLORMAP_JET
    )

    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    overlay = heatmap * 0.4 + (original_img * 255)

    overlay = np.clip(overlay / 255.0, 0, 1)

    plt.figure(figsize=(10,4))

    plt.subplot(1,3,1)
    plt.imshow(original_img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(cam, cmap='jet')
    plt.title("Grad-CAM")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis("off")

    plt.show()

In [ ]:
# pick any test image
idx = 0

img = X_test[idx]

# convert to tensor
input_tensor = torch.tensor(
    img.transpose(2,0,1),
    dtype=torch.float32
).unsqueeze(0).to(DEVICE)

# generate cam
cam = gradcam.generate(input_tensor)

# visualize
visualize_gradcam(img, cam)

In [ ]:
for i in range(5):
    img = X_test[i]

    input_tensor = torch.tensor(
        img.transpose(2,0,1),
        dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)

    cam = gradcam.generate(input_tensor)

    visualize_gradcam(img, cam)

In [ ]:
print("Feature variance:", np.var(Xh_train, axis=0).mean())
print(Xh_train.shape)
print(np.mean(Xh_train), np.std(Xh_train))

In [ ]:
dataset_path = "/kaggle/input/datasets/thilak02/breast-cancer-detection-using-thermography/BCD_Dataset"

print(os.listdir(dataset_path))

In [ ]:
IMG_SIZE = 224
images = []
labels = []

# -------- LOAD NORMAL IMAGES --------
normal_path = os.path.join(dataset_path, "normal")

for img in os.listdir(normal_path):

    img_path = os.path.join(normal_path, img)

    image = cv2.imread(img_path)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    images.append(image)
    labels.append(0)   # Normal = 0


# -------- LOAD SICK IMAGES --------
sick_path = os.path.join(dataset_path, "Sick")

for img in os.listdir(sick_path):

    img_path = os.path.join(sick_path, img)

    image = cv2.imread(img_path)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    images.append(image)
    labels.append(1)   # Abnormal = 1


# -------- CONVERT TO NUMPY ARRAYS --------
images = np.array(images)
labels = np.array(labels)

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)


# -------- NORMALIZE PIXEL VALUES --------
images = images / 255.0


# -------- DISPLAY SAMPLE IMAGE --------
plt.imshow(images[0])
plt.title(labels[0])
plt.axis("off")
plt.show()


# -------- TRAIN / TEST SPLIT --------
X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42
)


# -------- TRAIN / VALIDATION SPLIT --------
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)


# -------- CHECK DATASET SIZES --------
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
normal_count = len(os.listdir(normal_path))
sick_count = len(os.listdir(sick_path))

print("Number of Normal Images:", normal_count)
print("Number of Sick Images:", sick_count)
print("Total Images:", normal_count + sick_count)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# LOAD DATA
# =========================================================

X   = np.load("/kaggle/working/data/X.npy")
y   = np.load("/kaggle/working/data/y.npy")
raw = np.load("/kaggle/working/data/raw.npy")

n_classes = len(np.unique(y))
print("Data:", X.shape, raw.shape)

# =========================================================
# SPLIT
# =========================================================

X_train, X_test, y_train, y_test, rb_train, rb_test = train_test_split(
    X, y, raw, test_size=0.2, stratify=y, random_state=42
)

# =========================================================
# CLASS WEIGHTS (IMPORTANT FOR RECALL)
# =========================================================

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print("Class Weights:", class_weights_dict)

# =========================================================
# MODEL 1 — LIGHTGBM
# =========================================================

print("\n=== LightGBM ===")

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lgb_model = lgb.LGBMClassifier(
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight=class_weights_dict
)

lgb_model.fit(X_train_s, y_train)

pred_lgb = lgb_model.predict(X_test_s)
print(classification_report(y_test, pred_lgb))

# =========================================================
# MULTI-CHANNEL CNN INPUT
# =========================================================

def compute_entropy_channel(raw):
    win = 32
    ent = np.zeros_like(raw, dtype=np.float32)

    for i in range(raw.shape[0]):
        for j in range(0, raw.shape[1] - win, win):
            chunk = raw[i, j:j+win]
            hist, _ = np.histogram(chunk, bins=256, range=(0,256))
            p = hist / (hist.sum() + 1e-10)
            e = -np.sum(p * np.log2(p + 1e-10))
            ent[i, j:j+win] = e

    return ent / 8.0

def make_cnn_dataset(raw, labels):
    raw = raw.astype(np.float32)

    ch1 = (raw - 127.5) / 127.5
    ch2 = np.diff(raw, axis=1, prepend=raw[:, :1]) / 255.0
    ch3 = compute_entropy_channel(raw)

    X = np.stack([ch1, ch2, ch3], axis=1)

    return TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.long)
    )

train_ds = make_cnn_dataset(rb_train, y_train)
test_ds  = make_cnn_dataset(rb_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=128)

# =========================================================
# MODEL 2 — CNN + BiLSTM + ATTENTION
# =========================================================

print("\n=== CNN + BiLSTM + Attention ===")

class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Tanh(),
            nn.Linear(dim, 1)
        )

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return (weights * x).sum(dim=1)

class CipherNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(3, 64, 32, padding=16),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, 16, padding=8),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
        )

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.attention = Attention(256)

        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.permute(0, 2, 1)

        out, _ = self.lstm(x)
        attn_out = self.attention(out)

        return self.fc(attn_out)

model = CipherNet(n_classes).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
)

# TRAIN
EPOCHS = 25

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

# TEST
model.eval()
cnn_preds = []

with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(DEVICE)
        cnn_preds.extend(model(xb).argmax(1).cpu().numpy())

print("\nCNN Report:")
print(classification_report(y_test, cnn_preds))

# =========================================================
# HYBRID WITH ATTENTION FUSION
# =========================================================

print("\n=== HYBRID MODEL (Attention Fusion) ===")

def extract_embeddings(model, raw):
    model.eval()
    dataset = make_cnn_dataset(raw, np.zeros(len(raw)))
    loader = DataLoader(dataset, batch_size=256)

    embs = []

    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)

            x = model.conv(xb)
            x = x.permute(0, 2, 1)
            out, _ = model.lstm(x)
            attn_out = model.attention(out)

            embs.append(attn_out.cpu().numpy())

    return np.vstack(embs)

emb_train = extract_embeddings(model, rb_train)
emb_test  = extract_embeddings(model, rb_test)

# 🔥 ATTENTION FUSION (KEY IMPROVEMENT)
alpha = 0.6  # can tune this

Xh_train = np.hstack([alpha * X_train_s, (1 - alpha) * emb_train])
Xh_test  = np.hstack([alpha * X_test_s,  (1 - alpha) * emb_test])

# FINAL CLASSIFIER
hybrid = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=64,
    class_weight=class_weights_dict
)

hybrid.fit(Xh_train, y_train)

pred_hybrid = hybrid.predict(Xh_test)

print("\nHybrid Report:")
print(classification_report(y_test, pred_hybrid))

# =========================================================
# SUMMARY
# =========================================================

def summary(name, y_true, y_pred):
    print(f"{name}: acc={accuracy_score(y_true,y_pred):.4f}, "
          f"f1={f1_score(y_true,y_pred,average='macro'):.4f}")

print("\n=== FINAL SUMMARY ===")
summary("LightGBM", y_test, pred_lgb)
summary("CNN+Attention", y_test, cnn_preds)
summary("Hybrid", y_test, pred_hybrid)

In [ ]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
for layer in base_model.layers:
    layer.trainable = False
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation="relu")(x)

x = Dropout(0.5)(x)

output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5)

cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test,y_pred))

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

from skimage.feature import hog
from skimage.color import rgb2gray

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model

# ---------------------------
# 1 LOAD DATASET
# ---------------------------

IMG_SIZE = 224
images = []
labels = []

normal_path = os.path.join(dataset_path, "normal")

for img in os.listdir(normal_path):

    img_path = os.path.join(normal_path, img)

    image = cv2.imread(img_path)
    image = cv2.resize(image,(IMG_SIZE,IMG_SIZE))

    images.append(image)
    labels.append(0)

sick_path = os.path.join(dataset_path,"Sick")

for img in os.listdir(sick_path):

    img_path = os.path.join(sick_path,img)

    image = cv2.imread(img_path)
    image = cv2.resize(image,(IMG_SIZE,IMG_SIZE))

    images.append(image)
    labels.append(1)

images = np.array(images)
labels = np.array(labels)

print("Dataset:",images.shape)

# normalize
images = images / 255.0


# ---------------------------
# 2 TRAIN TEST SPLIT
# ---------------------------

X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42
)


# ---------------------------
# 3 DEEP FEATURE EXTRACTION
# ---------------------------

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

feature_extractor = Model(
    inputs=base_model.input,
    outputs=base_model.output
)

deep_features = feature_extractor.predict(images)

deep_features = deep_features.reshape(deep_features.shape[0], -1)

print("Deep Features:",deep_features.shape)


# ---------------------------
# 4 HOG FEATURE EXTRACTION
# ---------------------------

hog_features = []

for img in images:

    gray = rgb2gray(img)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16,16),
        cells_per_block=(2,2)
    )

    hog_features.append(features)

hog_features = np.array(hog_features)

print("HOG Features:",hog_features.shape)


# ---------------------------
# 5 FEATURE FUSION
# ---------------------------

combined_features = np.concatenate(
    (deep_features, hog_features),
    axis=1
)

print("Combined Features:",combined_features.shape)


# ---------------------------
# 6 PCA DIMENSION REDUCTION
# ---------------------------

pca = PCA(n_components=200)

X_pca = pca.fit_transform(combined_features)

print("After PCA:",X_pca.shape)


# ---------------------------
# 7 TRAIN TEST SPLIT AGAIN
# ---------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_pca,
    labels,
    test_size=0.2,
    random_state=42
)


# ---------------------------
# 8 SVM CLASSIFIER
# ---------------------------

model = SVC(kernel="rbf")

model.fit(X_train,y_train)

predictions = model.predict(X_test)


# ---------------------------
# 9 EVALUATION
# ---------------------------

accuracy = accuracy_score(y_test,predictions)

print("Hybrid Model Accuracy:",accuracy)

print(classification_report(y_test,predictions))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
normal_count = np.sum(labels == 0)
sick_count = np.sum(labels == 1)

print("Normal Images:", normal_count)
print("Sick Images:", sick_count)
classes = ["Normal", "Sick"]
counts = [normal_count, sick_count]

plt.bar(classes, counts)
plt.title("Dataset Class Distribution")
plt.ylabel("Number of Images")
plt.show()

In [ ]:
# ===============================
# 1. IMPORT LIBRARIES
# ===============================
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

from skimage.feature import hog
from skimage.color import rgb2gray

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ===============================
# 2. DATASET PATH
# ===============================
dataset_path = "/kaggle/input/datasets/thilak02/breast-cancer-detection-using-thermography/BCD_Dataset"

normal_path = os.path.join(dataset_path, "normal")
sick_path = os.path.join(dataset_path, "Sick")


# ===============================
# 3. DATA AUGMENTATION
# ===============================
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

IMG_SIZE = 224
images = []
labels = []


# ===============================
# 4. LOAD NORMAL IMAGES
# ===============================
for img in os.listdir(normal_path):

    if not img.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    img_path = os.path.join(normal_path, img)

    image = cv2.imread(img_path)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    images.append(image)
    labels.append(0)

    image_exp = image.reshape((1,) + image.shape)

    i = 0
    for batch in datagen.flow(image_exp, batch_size=1):
        images.append(batch[0].astype('uint8'))
        labels.append(0)
        i += 1
        if i == 2:
            break


# ===============================
# 5. LOAD SICK IMAGES
# ===============================
for img in os.listdir(sick_path):

    if not img.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    img_path = os.path.join(sick_path, img)

    image = cv2.imread(img_path)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    images.append(image)
    labels.append(1)

    image_exp = image.reshape((1,) + image.shape)

    i = 0
    for batch in datagen.flow(image_exp, batch_size=1):
        images.append(batch[0].astype('uint8'))
        labels.append(1)
        i += 1
        if i == 2:
            break


# ===============================
# 6. CONVERT + NORMALIZE
# ===============================
images = np.array(images)
labels = np.array(labels)

print("Dataset shape:", images.shape)

images = images / 255.0


# ===============================
# 7. SHOW SAMPLE IMAGES
# ===============================
normal_indices = np.where(labels == 0)[0][:3]
sick_indices = np.where(labels == 1)[0][:3]

plt.figure(figsize=(10,5))

for i, idx in enumerate(normal_indices):
    plt.subplot(2,3,i+1)
    plt.imshow(images[idx])
    plt.title("Normal")
    plt.axis("off")

for i, idx in enumerate(sick_indices):
    plt.subplot(2,3,i+4)
    plt.imshow(images[idx])
    plt.title("Sick")
    plt.axis("off")

plt.tight_layout()
plt.show()


# ===============================
# 8. DEEP FEATURES (ResNet50)
# ===============================
try:
    base_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224,224,3)
    )
    print("Using pretrained weights")
except:
    print("No internet, using random weights")
    base_model = ResNet50(
        weights=None,
        include_top=False,
        input_shape=(224,224,3)
    )

feature_extractor = Model(
    inputs=base_model.input,
    outputs=base_model.output
)

deep_features = feature_extractor.predict(images)
deep_features = deep_features.reshape(deep_features.shape[0], -1)

print("Deep Features:", deep_features.shape)


# ===============================
# 9. HOG FEATURES
# ===============================
hog_features = []

for img in images:
    gray = rgb2gray(img)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16,16),
        cells_per_block=(2,2)
    )

    hog_features.append(features)

hog_features = np.array(hog_features)

print("HOG Features:", hog_features.shape)


# ===============================
# 10. FEATURE FUSION
# ===============================
combined_features = np.concatenate((deep_features, hog_features), axis=1)

print("Combined Features:", combined_features.shape)


# ===============================
# 11. PCA
# ===============================
pca = PCA(n_components=200)

X_pca = pca.fit_transform(combined_features)

print("After PCA:", X_pca.shape)


# ===============================
# 12. TRAIN TEST SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X_pca,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)


# ===============================
# 13. XGBOOST MODEL
# ===============================
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)


# ===============================
# 14. RESULTS
# ===============================
accuracy = accuracy_score(y_test, predictions)

print("\nFinal Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, predictions))

Few Shot Ensemble


In [ ]:
pip install torch torchvision

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

DATA_DIR = "/kaggle/input/datasets/thilak02/breast-cancer-detection-using-thermography/BCD_Dataset"

# ================= DATASET =================
class BreastDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


# Load all samples
samples = []
classes = {'normal': 0, 'Sick': 1}

for cls in classes:
    path = os.path.join(DATA_DIR, cls)
    for img in os.listdir(path):
        samples.append((os.path.join(path, img), classes[cls]))

# ================= SPLIT =================
train_samples, val_samples = train_test_split(samples, test_size=0.2, stratify=[s[1] for s in samples])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

train_dataset = BreastDataset(train_samples, transform)
val_dataset = BreastDataset(val_samples, transform)

# ================= SAMPLER =================
labels = [label for _, label in train_samples]
class_counts = np.bincount(labels)
weights = 1. / class_counts
sample_weights = [weights[label] for label in labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# ================= MODELS =================
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet18(pretrained=True)
        self.model.fc = nn.Linear(512, 2)

    def forward(self, x):
        return self.model(x)


class ProtoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = models.resnet18(pretrained=True)
        self.encoder.fc = nn.Identity()

    def forward(self, x):
        return self.encoder(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn_model = CNNModel().to(device)
proto_model = ProtoNet().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-4)

# ================= TRAIN =================
for epoch in range(10):
    cnn_model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = cnn_model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# ================= PROTOTYPES =================
def get_prototypes(model, loader):
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            emb = model(imgs)
            features.append(emb.cpu())
            labels.append(lbls)

    features = torch.cat(features)
    labels = torch.cat(labels)

    prototypes = []
    for c in torch.unique(labels):
        prototypes.append(features[labels == c].mean(0))

    return torch.stack(prototypes)

# ================= ENSEMBLE =================
def ensemble_predict(cnn_model, proto_model, loader, threshold=0.4):
    cnn_model.eval()
    proto_model.eval()

    prototypes = get_prototypes(proto_model, loader).to(device)

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)

            # CNN
            cnn_out = cnn_model(imgs)
            cnn_prob = torch.softmax(cnn_out, dim=1)

            # Proto
            emb = proto_model(imgs)
            dists = torch.cdist(emb, prototypes)
            proto_prob = torch.softmax(-dists, dim=1)

            # Combine
            final_prob = 0.6 * cnn_prob + 0.4 * proto_prob

            cancer_prob = final_prob[:, 1]
            final_pred = (cancer_prob > threshold).long()

            all_preds.extend(final_pred.cpu().numpy())
            all_labels.extend(labels.numpy())

    return all_labels, all_preds

# ================= EVALUATE =================
y_true, y_pred = ensemble_predict(cnn_model, proto_model, val_loader, threshold=0.4)

print("Accuracy:", accuracy_score(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

SnapShot ensemble

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
DATA_DIR = "/kaggle/input/datasets/thilak02/breast-cancer-detection-using-thermography/BCD_Dataset"

class BreastDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


samples = []
classes = {'normal': 0, 'Sick': 1}

for cls in classes:
    path = os.path.join(DATA_DIR, cls)
    for img in os.listdir(path):
        samples.append((os.path.join(path, img), classes[cls]))

train_samples, val_samples = train_test_split(
    samples, test_size=0.2, stratify=[s[1] for s in samples]
)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

labels = [label for _, label in train_samples]
class_counts = np.bincount(labels)
weights = 1. / class_counts
sample_weights = [weights[label] for label in labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_dataset = BreastDataset(train_samples, transform)
val_dataset = BreastDataset(val_samples, transform)

train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet18(pretrained=True)
        self.model.fc = nn.Linear(512, 2)

    def forward(self, x):
        return self.model(x)


class ProtoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = models.resnet18(pretrained=True)
        self.encoder.fc = nn.Identity()

    def forward(self, x):
        return self.encoder(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn_model = CNNModel().to(device)
proto_model = ProtoNet().to(device)

optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=3,   # restart every 3 epochs
    T_mult=1
)



snapshots = []

EPOCHS = 12

for epoch in range(EPOCHS):
    cnn_model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = cnn_model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    # 🔥 Save snapshot at restart points
    if (epoch + 1) % 3 == 0:
        snap = CNNModel().to(device)
        snap.load_state_dict(cnn_model.state_dict())
        snapshots.append(snap)
        print(f"Snapshot saved at epoch {epoch+1}")




def get_prototypes(model, loader):
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            emb = model(imgs)
            features.append(emb.cpu())
            labels.append(lbls)

    features = torch.cat(features)
    labels = torch.cat(labels)

    prototypes = []
    for c in torch.unique(labels):
        prototypes.append(features[labels == c].mean(0))

    return torch.stack(prototypes)



def snapshot_ensemble_predict(snapshots, proto_model, loader, threshold=0.4):
    proto_model.eval()
    prototypes = get_prototypes(proto_model, loader).to(device)

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)

            # 🔥 Snapshot predictions
            snap_probs = []

            for model in snapshots:
                model.eval()
                out = model(imgs)
                prob = torch.softmax(out, dim=1)
                snap_probs.append(prob)

            avg_snap_prob = torch.mean(torch.stack(snap_probs), dim=0)

            # Proto predictions
            emb = proto_model(imgs)
            dists = torch.cdist(emb, prototypes)
            proto_prob = torch.softmax(-dists, dim=1)

            # Final ensemble
            final_prob = 0.7 * avg_snap_prob + 0.3 * proto_prob

            cancer_prob = final_prob[:, 1]
            final_pred = (cancer_prob > threshold).long()

            all_preds.extend(final_pred.cpu().numpy())
            all_labels.extend(labels.numpy())

    return all_labels, all_preds

y_true, y_pred = snapshot_ensemble_predict(
    snapshots,
    proto_model,
    val_loader,
    threshold=0.4
)

print("Accuracy:", accuracy_score(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

✅ Use multiple SVM kernels (linear, rbf, poly, sigmoid)
✅ Compare CNN vs SVM vs Ensemble
✅ Add GridSearch tuning
✅ Generate a comparison table (for report/viva)

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

DATA_DIR = "/kaggle/input/datasets/thilak02/breast-cancer-detection-using-thermography/BCD_Dataset"

class BreastDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


samples = []
classes = {'normal': 0, 'Sick': 1}

for cls in classes:
    path = os.path.join(DATA_DIR, cls)
    for img in os.listdir(path):
        samples.append((os.path.join(path, img), classes[cls]))

train_samples, val_samples = train_test_split(
    samples, test_size=0.2, stratify=[s[1] for s in samples]
)



transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = BreastDataset(train_samples, transform)
val_dataset = BreastDataset(val_samples, transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = models.resnet18(pretrained=True)
        self.base.fc = nn.Identity()  # remove classifier

    def forward(self, x):
        return self.base(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn_model = CNNModel().to(device)



def extract_features(model, loader):
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            emb = model(imgs)
            features.append(emb.cpu().numpy())
            labels.append(lbls.numpy())

    return np.vstack(features), np.hstack(labels)

X_train, y_train = extract_features(cnn_model, train_loader)
X_val, y_val = extract_features(cnn_model, val_loader)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)



kernels = ['linear', 'rbf', 'poly', 'sigmoid']
svm_results = {}

for kernel in kernels:
    print(f"\nTraining SVM ({kernel})...")

    model = SVC(kernel=kernel, probability=True)

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    acc = accuracy_score(y_val, preds)
    svm_results[kernel] = (model, acc)

    print(f"{kernel} Accuracy:", acc)


param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.001]
}

grid = GridSearchCV(SVC(kernel='rbf', probability=True), param_grid, cv=3)
grid.fit(X_train, y_train)

best_svm = grid.best_estimator_

print("\nBest SVM (RBF tuned):", grid.best_params_)



classifier = nn.Linear(512, 2).to(device)
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Train simple classifier
for epoch in range(5):
    classifier.train()
    total_loss = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        feats = cnn_model(imgs)
        outputs = classifier(feats)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"CNN Epoch {epoch+1}, Loss: {total_loss:.4f}")



def cnn_predict():
    classifier.eval()
    preds = []

    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(device)
            feats = cnn_model(imgs)
            out = classifier(feats)
            pred = torch.argmax(out, dim=1)
            preds.extend(pred.cpu().numpy())

    return preds

cnn_preds = cnn_predict()
cnn_acc = accuracy_score(y_val, cnn_preds)



svm_prob = best_svm.predict_proba(X_val)

cnn_probs = []
classifier.eval()

with torch.no_grad():
    for imgs, _ in val_loader:
        imgs = imgs.to(device)
        feats = cnn_model(imgs)
        out = classifier(feats)
        prob = torch.softmax(out, dim=1)
        cnn_probs.append(prob.cpu().numpy())

cnn_prob = np.vstack(cnn_probs)

# 🔥 Ensemble
final_prob = 0.6 * cnn_prob + 0.4 * svm_prob
ensemble_pred = np.argmax(final_prob, axis=1)

ensemble_acc = accuracy_score(y_val, ensemble_pred)



print("\n===== MODEL COMPARISON =====")

print(f"CNN Accuracy: {cnn_acc:.4f}")

for k in svm_results:
    print(f"SVM ({k}) Accuracy: {svm_results[k][1]:.4f}")

print(f"Tuned SVM (RBF) Accuracy: {accuracy_score(y_val, best_svm.predict(X_val)):.4f}")

print(f"Ensemble (CNN + SVM) Accuracy: {ensemble_acc:.4f}")



In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

def get_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred)
    }

results = []

# CNN
metrics = get_metrics(y_val, cnn_preds)
metrics["Model"] = "CNN"
results.append(metrics)

# SVM kernels
for k in svm_results:
    preds = svm_results[k][0].predict(X_val)
    metrics = get_metrics(y_val, preds)
    metrics["Model"] = f"SVM ({k})"
    results.append(metrics)

# Tuned SVM
metrics = get_metrics(y_val, best_svm.predict(X_val))
metrics["Model"] = "SVM (RBF Tuned)"
results.append(metrics)

# Ensemble
metrics = get_metrics(y_val, ensemble_pred)
metrics["Model"] = "CNN + SVM Ensemble"
results.append(metrics)

df = pd.DataFrame(results)
print(df)